# Vision-Transformer CRP — Walkthrough

Visual end-to-end exploration of the four ViT concept-detector classes added in this fork. The notebook is **visualisation-first**: every section produces a comparative grid of heatmaps, with numeric relevance shown as subplot labels rather than printed tables.

## Tour

1. **HeadConcept atlas** — for every layer in the layer-of-interest list, one row × `num_heads` cells, sorted by ascending head id. Identifies which heads carry the target-class signal at each depth.
2. **KQVHeadConcept atlas** — same per-layer layout but **3 rows (K, Q, V) × heads** so the K/Q/V triple of every head is column-aligned for direct comparison.
3. **HeadDim closeup** — fixes one (layer, head), shows the top-K `HeadDimConcept` heatmaps in id order. Per-layer figures so you can track how the same head's fine structure shifts through depth.
4. **KQVHeadDim closeup** — same fix, **3 rows × top-K dims** (K[d]/Q[d]/V[d] column-aligned).
5. **Conditional layer cascade** — backward CRP through a configurable layer list. At the deepest layer pick the top-K heads on the target class; at each shallower layer pick the top-K *conditioned on* the deeper-layer selection. Every picked head renders next to its top reference samples from the FV index.
6. **Reference samples** — for each of the four granularities at the chosen mid-layer, render the top concepts' representative images side-by-side with their conditional heatmaps on the target.

**Concept cheat sheet** (two orthogonal granularity axes — *split by K/Q/V?* and *split by head_dim?*):

| Class | Tap | Granularity | `attribute()` shape |
|---|---|---|---|
| `HeadConcept`        | `attn_out_tap` | per head (output tokens)              | `(B, num_heads)`              |
| `HeadDimConcept`     | `attn_out_tap` | per `(head, dim)` (output tokens)     | `(B, num_heads, head_dim)`    |
| `KQVHeadConcept`     | `qkv_tap`      | per `(part, head)` (K/Q/V projections) | `(B, 3, num_heads)`         |
| `KQVHeadDimConcept`  | `qkv_tap`      | per `(part, head, dim)` (K/Q/V projections) | `(B, 3, num_heads, head_dim)` |

Theory: AttnLRP (Achtibat et al., ICML 2024; [arXiv 2402.05602](https://arxiv.org/abs/2402.05602)) on top of CRP (Achtibat et al., Nature MI 2023; [arXiv 2206.03208](https://arxiv.org/abs/2206.03208)).

## 1. Setup

Imports and path bootstrap. `experiments/` lives on `sys.path` so the shared `viz.py` plotting helpers and `datasets.py` loader resolve.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import timm
from timm.data import resolve_data_config, create_transform

from crp.attribution import CondAttribution
from crp.attention_concepts import (
    HeadConcept, HeadDimConcept, KQVHeadConcept, KQVHeadDimConcept, PARTS,
)
from crp.transformer_patches import (
    AttnLRPEpsilonComposite, AttnLRPGammaComposite,
)
from crp.visualization import FeatureVisualization

# Repo-root locator: walks up from CWD until pyproject.toml is found.
def _repo_root():
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / 'pyproject.toml').is_file():
            return p
        p = p.parent
    raise RuntimeError('repo root with pyproject.toml not found above CWD')
REPO_ROOT = _repo_root()
DATA_DIR = REPO_ROOT / 'data'
FV_ROOT = DATA_DIR / 'feature_visualization'
DATA_DIR.mkdir(parents=True, exist_ok=True)
FV_ROOT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT / 'experiments'))
from datasets import load as load_dataset, IMAGENETTE_CLASS_NAMES  # noqa: E402
from viz import (  # noqa: E402
    CONCEPT_CLASSES, denormalize,
    plot_head_atlas, plot_kqv_head_atlas,
    plot_head_dim_closeup, plot_kqv_head_dim_closeup,
    plot_conditional_cascade, plot_reference_samples,
)

torch.set_grad_enabled(True)
print('torch', torch.__version__, '| timm', timm.__version__)

## 2. Configuration

All knobs in one place. Override here for a quicker run on a smaller model, a different dataset, or a different image.

* `MODEL_NAME` — `vit_base_patch16_224` (86 M, 12 layers, 12 heads, head_dim 64) is the default. `vit_small_patch16_224` (22 M, 6 heads) and `vit_tiny_patch16_224` (5 M, 3 heads) run faster.
* `DATASET_NAME` — `'imagenette'` (10-class, 98 MB, auto-downloaded) is the default. Switch to `'imagenet_val'` once `data/imagenet_val/` is populated; see `experiments/datasets.py`.
* `LAYERS_OF_INTEREST` — layers used by the per-layer atlas / closeup plotters. Mid-network layers (5–9 on a 12-layer ViT) tend to carry the most class-relevant structure.
* `MID_LAYER` — single layer used to pick the closeup head and to build the §13 reference-sample indices.
* `CASCADE_LAYERS` — backward-cascade layer list (deepest first). Default `[11, 8, 5, 2]` covers the network depth without indexing an FV at every layer.
* `CLOSEUP_HEAD` — head id for §9/§10 closeups. `None` picks the top-1 HeadConcept head at `MID_LAYER` automatically.
* `CELL_SIZE` — per-panel width in inches (controls figure resolution).
* `USE_GAMMA` — γ-LRP is available but **not** the default. On `vit_base` the γ rule stack inflates relevance ~10¹⁶× (see [`CURRENT_STATE.md`](../../CURRENT_STATE.md) Milestone D), making heatmaps numerically degenerate. Plain ε-LRP gives clean, interpretable heatmaps.

In [ ]:
MODEL_NAME = 'vit_base_patch16_224'   # 'vit_small_patch16_224' / 'vit_tiny_patch16_224'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

DATASET_NAME = 'imagenette'           # 'imagenette' | 'imagenet_val'
NUM_SAMPLES = 64                      # FV index size — bump up for cleaner reference samples

LAYERS_OF_INTEREST = [3, 6, 9, 11]    # for atlases / closeups
MID_LAYER = 6
CASCADE_LAYERS = [11, 8, 5, 2]        # deepest first

TOP_K_HEAD = None                     # None → all heads in atlas; else top-K by id
TOP_K_DIM = 8                         # dims per head in closeups
TOP_K_CASCADE = 4                     # heads picked per cascade layer
N_REFS = 4                            # FV reference samples per concept
CLOSEUP_HEAD: int | None = None       # None → auto-pick top-1 head at MID_LAYER

CELL_SIZE = 1.4                       # inches per panel; bump for higher resolution

TARGET_INDEX = None                   # int → that index; None → random under RANDOM_SEED
RANDOM_SEED = 0

USE_GAMMA = False                     # γ-LRP catastrophically inflates on vit_base
GAMMA = 0.25
EPSILON = 1e-6

print(f'device  : {DEVICE}')
print(f'model   : {MODEL_NAME}')
print(f'dataset : {DATASET_NAME}')
print(f'layers  : {LAYERS_OF_INTEREST}  (mid={MID_LAYER})')
print(f'cascade : {CASCADE_LAYERS}')
print(f'rule    : {("γ-LRP, γ=" + str(GAMMA)) if USE_GAMMA else "ε-LRP"}')

## 3. Dataset, model, and composite

`load_dataset` auto-downloads Imagenette on first call (~98 MB). The composite bundles `TimmViTCanonizer`, which installs the `qkv_tap` and `attn_out_tap` `nn.Identity` submodules and swaps `forward` on the timm Attention to embed AttnLRP's autograd rules — all scoped to `composite.context()` and reverted on exit.

In [ ]:
if DATASET_NAME == 'imagenette':
    n_per_class = max(1, NUM_SAMPLES // 10)
    classes = None
elif DATASET_NAME == 'imagenet_val':
    n_per_class = max(1, NUM_SAMPLES // 1000)
    classes = None
else:
    raise ValueError(DATASET_NAME)

model = timm.create_model(MODEL_NAME, pretrained=True).eval().to(DEVICE)
cfg = resolve_data_config({}, model=model)
preprocess_fn = create_transform(**cfg)

dataset = load_dataset(
    DATASET_NAME, root=DATA_DIR, n_per_class=n_per_class,
    classes=classes, seed=RANDOM_SEED,
    transform=preprocess_fn,
)
print(f'{dataset.name}: {len(dataset)} images, {dataset.num_classes} classes')

if USE_GAMMA:
    composite = AttnLRPGammaComposite(gamma=GAMMA, epsilon=EPSILON)
else:
    composite = AttnLRPEpsilonComposite(epsilon=EPSILON)
attribution = CondAttribution(model, device=torch.device(DEVICE))

NUM_HEADS = model.blocks[MID_LAYER].attn.num_heads
HEAD_DIM = model.blocks[MID_LAYER].attn.head_dim
print(f'{type(composite).__name__}: num_heads={NUM_HEADS}, head_dim={HEAD_DIM}')

## 4. Pick a target image

Attribute under the image's true class. Re-run with a different `RANDOM_SEED` to pick another image.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
target_idx = TARGET_INDEX if TARGET_INDEX is not None else int(rng.integers(0, len(dataset)))
target_data, target_class = dataset[target_idx]
target_pre = target_data.unsqueeze(0).to(DEVICE).requires_grad_(True)
class_name = IMAGENETTE_CLASS_NAMES.get(target_class, str(target_class))

fig, ax = plt.subplots(1, 1, figsize=(3, 3))
ax.imshow(denormalize(target_pre, model))
ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f'idx {target_idx}  •  cls {target_class}  •  {class_name}', fontsize=10)
plt.show()

## 5. HeadConcept atlas (one figure per layer)

For every layer in `LAYERS_OF_INTEREST`, render the input + heatmap side-by-side for **every head**, sorted by ascending head id. The subplot title carries the concept's relevance score on the target class. Look for: (a) which heads at each depth attend to the salient object; (b) how the focus shifts depth-wise.

In [ ]:
for fig in plot_head_atlas(
    target_pre, model, attribution, composite,
    layers=LAYERS_OF_INTEREST, target_class=target_class,
    top_k=TOP_K_HEAD, cell_size=CELL_SIZE,
    suptitle_prefix=class_name,
):
    plt.show()

## 6. KQVHeadConcept atlas — K/Q/V coupled per head

For every layer in `LAYERS_OF_INTEREST`, **3 rows (K, Q, V) × heads** (by id). Column alignment lets you compare the K/Q/V triple of one head directly. K often shows *where* the head attends, V *what* it carries, and Q the query dependence.

In [ ]:
for fig in plot_kqv_head_atlas(
    target_pre, model, attribution, composite,
    layers=LAYERS_OF_INTEREST, target_class=target_class,
    top_k=TOP_K_HEAD, cell_size=CELL_SIZE,
    suptitle_prefix=class_name,
):
    plt.show()

## 7. Pick a head for the per-dim closeups

The HeadDim / KQVHeadDim closeups need a single head to focus on. By default we pick the top-1 HeadConcept head at `MID_LAYER` — override `CLOSEUP_HEAD` in §2 to pin a different one.

In [ ]:
from viz import _per_concept_scores, _layer_name_for  # noqa: F401

if CLOSEUP_HEAD is None:
    _hc = HeadConcept(model)
    _layer = _layer_name_for(MID_LAYER, _hc)
    _scores = _per_concept_scores(
        attribution, composite, target_pre, _layer, _hc, target_class,
    )
    closeup_head = int(torch.argmax(_scores.abs()))
    print(f'auto-picked head {closeup_head}  '
          f'(MID_LAYER={MID_LAYER} score={_scores[closeup_head].item():+.3f})')
else:
    closeup_head = CLOSEUP_HEAD
    print(f'using configured CLOSEUP_HEAD = {closeup_head}')

## 8. HeadDimConcept closeup — per-dim fine structure within one head

For `closeup_head`, render the top-`TOP_K_DIM` head_dim heatmaps (by |relevance|, then sorted by ascending dim id) for every layer in `LAYERS_OF_INTEREST`. Each row is one layer; columns are individual dimensions of that head's output tokens.

In [ ]:
for fig in plot_head_dim_closeup(
    target_pre, model, attribution, composite,
    layers=LAYERS_OF_INTEREST, head_id=closeup_head,
    target_class=target_class, top_k=TOP_K_DIM, cell_size=CELL_SIZE,
    suptitle_prefix=class_name,
):
    plt.show()

## 9. KQVHeadDimConcept closeup — K/Q/V × dims for one head

Same fix (one head per layer), but **3 rows (K, Q, V) × top-`TOP_K_DIM` dims** (by aggregate KQV |relevance|, displayed in dim id order). K[d]/Q[d]/V[d] are column-aligned so you can compare the three projections of the same dim directly.

In [ ]:
for fig in plot_kqv_head_dim_closeup(
    target_pre, model, attribution, composite,
    layers=LAYERS_OF_INTEREST, head_id=closeup_head,
    target_class=target_class, top_k=TOP_K_DIM, cell_size=CELL_SIZE,
    suptitle_prefix=class_name,
):
    plt.show()

## 10. Build a multi-layer FV index for the cascade

The conditional cascade in §11 needs an FV index of `HeadConcept` at **every** layer in `CASCADE_LAYERS`. A single `FeatureVisualization` instance with a multi-layer `layer_map` indexes them all in one pass (one forward+backward per dataset image, multiple recorded layers). Cached at `data/feature_visualization/cascade/` after first run.

In [ ]:
_hc = HeadConcept(model)
cascade_layer_names = [_layer_name_for(L, _hc) for L in CASCADE_LAYERS]
fv_cascade = FeatureVisualization(
    attribution, dataset,
    layer_map={ln: _hc for ln in cascade_layer_names},
    preprocess_fn=preprocess_fn,
    path=str(FV_ROOT / 'cascade'),
    device=torch.device(DEVICE),
)
for L, ln in zip(CASCADE_LAYERS, cascade_layer_names):
    print(f'  L{L}: {ln}')

In [ ]:
%%time
rel_dir = FV_ROOT / 'cascade' / 'RelMax_sum_normed'
if rel_dir.is_dir() and any(rel_dir.glob('*.npy')):
    print(f'cached index found at {rel_dir} — skipping fv_cascade.run()')
else:
    print(f'running multi-layer FV index across {len(CASCADE_LAYERS)} layers...')
    fv_cascade.run(composite, 0, len(dataset), batch_size=8, checkpoint=10000)
    print('done.')

## 11. Conditional layer cascade

Backward conditional CRP through `CASCADE_LAYERS`. At the deepest layer the top-`TOP_K_CASCADE` HeadConcept heads are picked under `{y: target_class}`. At each shallower layer the top heads are picked under `{y: target_class} ∪ {deeper_layer_taps: previously_picked_heads}`, i.e. *conditioned on* the union of selections at deeper layers.

The figure has one row per cascade layer (late → early). Each row shows `TOP_K_CASCADE` head-blocks; each block is the target image with this head's conditional heatmap (1 panel) plus the head's top-`N_REFS` reference samples from the FV index (`N_REFS` panels).

In [ ]:
fig, selected = plot_conditional_cascade(
    target_pre, model, attribution, composite, fv_cascade,
    layers=CASCADE_LAYERS, target_class=target_class,
    top_k_per_layer=TOP_K_CASCADE, n_refs=N_REFS,
    cell_size=CELL_SIZE,
    suptitle=f'conditional cascade  •  {class_name}  •  '
             f'{type(composite).__name__}',
)
plt.show()
print('selected heads per layer:')
for L, heads in selected.items():
    print(f'  L{L}: {heads}')

## 12. Reference samples per granularity at MID_LAYER

For each of the four concept classes, build a separate FV index at `MID_LAYER`, then render the top-`TOP_K_CASCADE` concepts on the target image with their FV reference samples. Each row is one concept; columns are the target panel + `N_REFS` reference panels.

In [ ]:
fvs_mid = {}
concepts_mid = {}
for name, cls in CONCEPT_CLASSES.items():
    concept = cls(model)
    concepts_mid[name] = concept
    layer_name = _layer_name_for(MID_LAYER, concept)
    fvs_mid[name] = FeatureVisualization(
        attribution, dataset, layer_map={layer_name: concept},
        preprocess_fn=preprocess_fn,
        path=str(FV_ROOT / f'mid_{name}'),
        device=torch.device(DEVICE),
    )
for name in CONCEPT_CLASSES:
    print(f'  {name:13} → blocks.{MID_LAYER}.attn.{concepts_mid[name].tap_name}')

In [ ]:
%%time
for name, fv in fvs_mid.items():
    rel_dir = FV_ROOT / f'mid_{name}' / 'RelMax_sum_normed'
    if rel_dir.is_dir() and any(rel_dir.glob('*.npy')):
        print(f'[{name}] cached index — skipping')
        continue
    print(f'\n=== running FV index for {name!r} ===')
    fv.run(composite, 0, len(dataset), batch_size=8, checkpoint=10000)
print('\nall four indices ready.')

In [ ]:
fig = plot_reference_samples(
    fvs_mid['head'],
    concept_name='head', layer_idx=MID_LAYER,
    target_image=target_pre, target_class=target_class,
    model=model, attribution=attribution, composite=composite,
    n_top_concepts=TOP_K_CASCADE, n_refs_per_concept=N_REFS,
    cell_size=CELL_SIZE,
    suptitle=f'head  •  layer {MID_LAYER}  •  references',
)
plt.show()

In [ ]:
fig = plot_reference_samples(
    fvs_mid['head_dim'],
    concept_name='head_dim', layer_idx=MID_LAYER,
    target_image=target_pre, target_class=target_class,
    model=model, attribution=attribution, composite=composite,
    n_top_concepts=TOP_K_CASCADE, n_refs_per_concept=N_REFS,
    cell_size=CELL_SIZE,
    suptitle=f'head_dim  •  layer {MID_LAYER}  •  references',
)
plt.show()

In [ ]:
fig = plot_reference_samples(
    fvs_mid['kqv_head'],
    concept_name='kqv_head', layer_idx=MID_LAYER,
    target_image=target_pre, target_class=target_class,
    model=model, attribution=attribution, composite=composite,
    n_top_concepts=TOP_K_CASCADE, n_refs_per_concept=N_REFS,
    cell_size=CELL_SIZE,
    suptitle=f'kqv_head  •  layer {MID_LAYER}  •  references',
)
plt.show()

In [ ]:
fig = plot_reference_samples(
    fvs_mid['kqv_head_dim'],
    concept_name='kqv_head_dim', layer_idx=MID_LAYER,
    target_image=target_pre, target_class=target_class,
    model=model, attribution=attribution, composite=composite,
    n_top_concepts=TOP_K_CASCADE, n_refs_per_concept=N_REFS,
    cell_size=CELL_SIZE,
    suptitle=f'kqv_head_dim  •  layer {MID_LAYER}  •  references',
)
plt.show()

## 13. Notes

* Every panel is a *side-by-side concatenation* of the input image and the heatmap (no alpha overlay), keeping both vivid.
* Cells within a row are always ordered by ascending concept id (head id, then dim id). Top-K filtering selects which cells are shown but the order is deterministic.
* `CELL_SIZE` (§2) controls per-panel width in inches; the figure size scales with the number of cells in the layout.
* For the cleanest heatmaps on `vit_base`, leave `USE_GAMMA = False`. γ-LRP at γ=0.25 inflates relevance ~10¹⁶× on a 12-layer ViT (see `CURRENT_STATE.md` Milestone D), which matplotlib clips at the colour-bar extremes.
* Bump `NUM_SAMPLES` to 128 / 256 / … for cleaner reference samples (FV indexes more dataset images, top-N representatives become more consistent). The atlas / closeup sections don't depend on FV and run in <1 minute regardless.
* Quantitative benchmark numbers (Petsiuk deletion / insertion AUC, PA-LRP, residual-LRP) live in [`experiments.ipynb`](experiments.ipynb) and the milestone CSVs under `data/`.